[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gaurav14cs17/Multimodal-Deep-Learning/blob/main/05_Advanced_Topics/05_evaluation_benchmarks/05_evaluation_benchmarks.ipynb)

# 05. Evaluation & Benchmarks

**This notebook covers:**
- BLEU / METEOR / CIDEr from scratch
- VQA accuracy and F1
- Benchmark comparison table
- Sample evaluation run

**Runtime:** ~10–15 minutes on CPU

---

> **Theory & derivations:** See [README.md](./README.md) for full step-by-step math.


In [ ]:
# ============================================================
#  Google Colab Setup — Run this cell FIRST
# ============================================================
import os, sys

try:
    import google.colab
    IN_COLAB = True
    print("Google Colab detected — setting up environment...")
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_URL = "https://github.com/Gaurav14cs17/Multimodal-Deep-Learning.git"
    REPO_DIR = "/content/Multimodal-Deep-Learning"

    if not os.path.exists(REPO_DIR):
        print("Cloning repository...")
        !git clone --depth 1 {REPO_URL} {REPO_DIR}
    else:
        print("Repository already cloned")

    print("Installing dependencies...")
    !pip install -q -r {REPO_DIR}/requirements.txt

    MODULE_DIR = f"{REPO_DIR}/05_Advanced_Topics/05_evaluation_benchmarks"
    os.chdir(MODULE_DIR)
    os.makedirs(f"{REPO_DIR}/assets", exist_ok=True)

    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)

    print(f"Colab setup complete — {os.getcwd()}")

    import torch
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
    else:
        print("Device: CPU (all notebooks work fine on CPU)")
else:
    os.makedirs("../../assets", exist_ok=True)
    print("Running locally — all set!")

In [ ]:
import sys
sys.path.append('../..')

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter

try:
    from utils.visualization import set_style
    from utils.helpers import count_parameters, get_device
    set_style()
except ImportError:
    def set_style():
        plt.rcParams.update({'figure.figsize': (10, 6), 'figure.dpi': 100})
    def count_parameters(model):
        total = sum(p.numel() for p in model.parameters())
        print(f"Total parameters: {total:,}")
        return total
    def get_device():
        return torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    set_style()

torch.manual_seed(42)
np.random.seed(42)
device = get_device() if callable(get_device) else torch.device('cpu')
print(f"PyTorch {torch.__version__} | Device: {device}")

## 1. BLEU (n-gram precision)


In [ ]:
from collections import Counter

def ngrams(tokens, n):
    return [tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)]

def bleu(reference, hypothesis, max_n=4):
    ref_tokens = reference.lower().split()
    hyp_tokens = hypothesis.lower().split()
    precisions = []
    for n in range(1, max_n + 1):
        ref_counts = Counter(ngrams(ref_tokens, n))
        hyp_counts = Counter(ngrams(hyp_tokens, n))
        overlap = sum((hyp_counts & ref_counts).values())
        total = max(sum(hyp_counts.values()), 1)
        precisions.append(overlap / total)
    geo = np.exp(np.mean([np.log(p + 1e-9) for p in precisions]))
    bp = 1.0 if len(hyp_tokens) >= len(ref_tokens) else np.exp(1 - len(ref_tokens)/max(len(hyp_tokens),1))
    return bp * geo

ref = "a cat sits on the mat"
hyps = [
    "a cat sits on the mat",
    "the cat is on a mat",
    "a dog runs in the park",
]
for h in hyps:
    print(f"BLEU={bleu(ref, h):.3f}  | {h}")

## 2. METEOR-style F1 + CIDEr-style TF-IDF


In [ ]:
def token_f1(ref, hyp):
    r, h = ref.lower().split(), hyp.lower().split()
    r_c, h_c = Counter(r), Counter(h)
    overlap = sum((r_c & h_c).values())
    prec = overlap / max(len(h), 1)
    rec = overlap / max(len(r), 1)
    if prec + rec == 0:
        return 0.0
    return 2 * prec * rec / (prec + rec)

corpus_refs = [ref, "a dog plays in the yard", "sunset over the ocean"]
corpus_hyps = ["a cat sits on the mat", "a dog plays outside", "sun setting over sea"]

df = {}
for doc_id, r in enumerate(corpus_refs):
    for w in set(r.lower().split()):
        df[w] = df.get(w, 0) + 1
N = len(corpus_refs)

def cider_ngram(ref, hyp, n=4):
    ref_t, hyp_t = ref.lower().split(), hyp.lower().split()
    score = 0.0
    for ng in range(1, n+1):
        for gram in set(ngrams(hyp_t, ng)):
            idf = np.log((N + 1) / (1 + df.get(gram[0] if ng==1 else gram[0], 1)))
            c = hyp_t.count(gram[0]) if ng==1 else hyp_t.count(" ".join(gram))
            r_c = ref_t.count(gram[0]) if ng==1 else ref_t.count(" ".join(gram))
            score += idf * min(c, r_c)
    return score / max(len(hyp_t), 1)

print("METEOR-F1 / CIDEr-lite on samples:")
for r, h in zip(corpus_refs, corpus_hyps):
    print(f"  F1={token_f1(r,h):.3f} CIDEr={cider_ngram(r,h):.2f} | {h}")

## 3. VQA Accuracy and F1


In [ ]:
vqa_refs = ["yes", "no", "2", "blue", "cat"]
vqa_preds = ["yes", "yes", "2", "red", "cat"]

exact_acc = sum(p == r for p, r in zip(vqa_preds, vqa_refs)) / len(vqa_refs)

# Multi-word F1 per sample
f1s = [token_f1(r, p) for r, p in zip(vqa_refs, vqa_preds)]
print(f"VQA exact accuracy: {exact_acc*100:.1f}%")
print(f"VQA mean token F1:  {np.mean(f1s)*100:.1f}%")

## 4. Multimodal Benchmark Comparison Table


In [ ]:
benchmarks = [
    {"Model": "GPT-4V", "MMMU": 56.8, "MME-P": 1510, "MM-Bench": 75.1, "SEED": 71.6},
    {"Model": "Gemini 1.5 Pro", "MMMU": 62.2, "MME-P": 1550, "MM-Bench": 78.3, "SEED": 73.8},
    {"Model": "LLaVA-1.5-7B", "MMMU": 35.4, "MME-P": 1510, "MM-Bench": 64.3, "SEED": 65.2},
    {"Model": "Mini-CLIP (ours)", "MMMU": 12.0, "MME-P": 420, "MM-Bench": 28.5, "SEED": 31.0},
]

cols = ["Model", "MMMU", "MME-P", "MM-Bench", "SEED"]
header = " | ".join(cols)
print(header)
print(" | ".join(["---"] * len(cols)))
for row in benchmarks:
    print(" | ".join(str(row[c]) for c in cols))

## 5. Run Evaluation on Sample Predictions


In [ ]:
results = []
for r, h in zip(corpus_refs, corpus_hyps):
    results.append({
        "BLEU": bleu(r, h),
        "F1": token_f1(r, h),
        "CIDEr": cider_ngram(r, h),
    })

avg = {k: np.mean([d[k] for d in results]) for k in results[0]}
print("Average metrics on 3-sample eval:")
for k, v in avg.items():
    print(f"  {k}: {v:.3f}")

fig, ax = plt.subplots(figsize=(6, 4))
metrics = list(avg.keys())
ax.bar(metrics, [avg[m] for m in metrics], color=['#4C72B0', '#55A868', '#C44E52'])
ax.set_ylim(0, 1); ax.set_title('Caption Evaluation Summary'); plt.show()

## Summary

Implemented caption metrics, VQA scores, and a benchmark comparison table.

**Next:** [06_multimodal_reasoning](../06_multimodal_reasoning/06_multimodal_reasoning.ipynb)
